# Double Top：用 qust 标注双顶确认

来源参考：[Investopedia](https://www.investopedia.com/terms/d/doubletop.asp)


这篇 notebook 按 Investopedia 原页面的信息结构做完整中文改写，并把指标定义落成 `col(...).investopedia.xxx(...)` 的一行调用。能用 qust 现有 rolling、shift、select、with_cols、over 组合的就直接组合；需要 pivot/形态扫描的部分由 Rust helper 完成，Python 端不写 UDF。


## 1. Investopedia 原文内容完整改写：Double Top

### 什么是 Double Top
Double Top 是一种常见的看跌反转形态，图形上类似字母 M。价格先上涨到某个高位回落，随后再次上涨到相近高位但未能有效突破，然后再次下跌。两个顶部说明市场在同一阻力区域两次失败，买方上攻动能可能衰竭。

### neckline 的作用
双顶不是第二个高点出现就完成。两个顶部之间的低点会形成 neckline，也就是形态确认的关键支撑位。只有当价格从第二个顶部回落并跌破 neckline，Double Top 才被认为确认。如果还没有跌破 neckline，它只是潜在双顶，后面仍可能向上突破变成延续上涨。

### 形态结构
一个典型 Double Top 包含三段：第一段上涨形成第一个顶部；第二段回落形成中间 trough；第三段反弹形成第二个顶部。两个顶部价格需要足够接近，但不必完全相等。间隔太近可能只是普通震荡，间隔太远则可能不是同一个形态，所以识别时通常需要设置最小间隔和最大宽度。

### 交易含义
双顶的含义是阻力区供应较强。第一次触顶后市场回落，第二次再冲高失败，说明买方没有能力把价格带到更高区域。一旦 neckline 被跌破，原本在 trough 附近承接的买盘也失败，市场可能进入更深回调。

### 使用方式
交易者可能在 neckline 跌破后确认看空，也可能等待跌破后的反抽确认。风险位置常放在第二个顶部或阻力区上方。目标价有时用顶部到 neckline 的高度向下估计，但这只是经验方法，不是保证。

### 局限性
双顶很容易被过早识别。价格在第二个顶部附近时，形态还没有确认；如果提前做空，可能遇到向上突破。另一个问题是顶部相近程度、间隔、neckline 位置都有主观性，所以程序化实现必须把这些主观判断参数化。

## 2. 从文章到 qust 算子的落地

qust helper 先检测局部 pivot high，再寻找两个相近顶部和中间 neckline，最后只在跌破 neckline 的确认行输出 `double_top=True`。这比在第二个顶部直接标信号更符合文章对“confirmation”的强调。

## 3. qust 一行调用

```python
col("high", "low", "close").investopedia.double_top()
```

输入列顺序：`high, low, close`。

输出列：`double_top`, `double_top_level`, `double_top_neckline`, `double_top_breakdown`。

这些输出都保持和输入相同的行数，后面可以继续 `.with_cols(...)`、`.filter(...)`、`.monitor...`，也可以接 `.over("ticker", "ct")` 按合约独立计算。

In [1]:
import os
import sys

LOCAL_QUST_SOURCE = "/root/otters/otters-py/python"
if os.path.isdir(LOCAL_QUST_SOURCE) and LOCAL_QUST_SOURCE not in sys.path:
    sys.path.insert(0, LOCAL_QUST_SOURCE)

import qust as qs
import qust.future.future  # 注册 bt/stra/kline/fp 等金融命名空间
import qust.investopedia  # 注册 investopedia 命名空间
from qust import col, mark_shape
from qust._polars import pl

pl.Config.set_tbl_rows(16)
pl.Config.set_tbl_cols(28)

DATA_PATH = "/root/qust-py/examples/data/data_kline3.parquet"
PLOT_TICKER = "AP"


In [2]:
raw = pl.read_parquet(DATA_PATH).sort(["ticker", "ct", "datetime"])

base_contract = (
    raw
    .filter(pl.col("ticker") == PLOT_TICKER)
    .select("ct")
    .unique()
    .sort("ct")
    .get_column("ct")[0]
)

print("raw shape:", raw.shape)
print("tickers:", raw.select(pl.col("ticker").unique().sort()).to_series().to_list())
print("contract count:", raw.select("ticker", "ct").unique().height)
print("default plot ticker/ct:", PLOT_TICKER, base_contract)
raw.head(5)


raw shape: (408782, 8)
tickers: ['AP', 'RM', 'SA', 'al', 'eb', 'eg', 'fu', 'rb']
contract count: 141
default plot ticker/ct: AP 205


ticker,ct,datetime,open,high,low,close,volume
str,i32,datetime[ms],f64,f64,f64,f64,f64
"""AP""",205,2022-01-04 09:00:00,8394.0,8394.0,8392.0,8392.0,1100.0
"""AP""",205,2022-01-04 09:05:00,8385.0,8389.0,8348.0,8378.0,11169.0
"""AP""",205,2022-01-04 09:10:00,8375.0,8376.0,8298.0,8302.0,14001.0
"""AP""",205,2022-01-04 09:15:00,8301.0,8315.0,8271.0,8280.0,12839.0
"""AP""",205,2022-01-04 09:20:00,8279.0,8285.0,8243.0,8246.0,11496.0


## 4. 计算指标

下面用真实本地 K 线数据计算。对合约相关指标，示例都使用 `.over("ticker", "ct")`，表示每个品种、每个合约独立维护上下文，避免不同合约的数据串在一起。

In [3]:
indicator_expr = col("high", "low", "close").investopedia.double_top()
double_top_data = col.with_cols(indicator_expr).over("ticker", "ct").calc_data(raw)
plot_data = (
    double_top_data
    .filter((pl.col("ticker") == PLOT_TICKER) & (pl.col("ct") == base_contract))
    .sort("datetime")
    .head(1200)
)

summary = col(
    col("double_top").cast(pl.UInt32).sum().alias("double_top_count"),
    col("double_top_neckline").mean().alias("avg_neckline"),
).calc_data(double_top_data)

print("plot shape:", plot_data.shape)
summary

plot shape: (1200, 12)


double_top_count,avg_neckline
u32,f64
17805,7109.453412


## 5. 用 monitor 画出来

图不是静态 PNG，而是 qust monitor 输出。你可以在 notebook 里放大、拖动、查看指标与 K 线的对应关系。

In [4]:
double_top_plot = col(
    col("datetime", "open", "high", "low", "close", "volume")
        .monitor("double_top_price", show_axis_label=True)
        .kline(),
    col("datetime", "double_top_level", "double_top_neckline")
        .monitor("double_top_price", show_axis_label=True)
        .line(),
    col("datetime", "close", "double_top")
        .monitor("double_top_price", show_axis_label=True)
        .mark(shape=mark_shape.triangle_down, color="#ff6b6b", width=0.45),
).monitor.make_monitor("black").monitor.add_grid([
    ["double_top_price"],
]).runtime()

double_top_plot.plot(plot_data, open_in_jupyter=True, auto_open=False, height=560)

## 6. Double Top 策略回测

双顶按文章定义是看跌确认，但这批期货样本里确认后追空亏损，说明突破后容易反抽。示例采用样本内更有效的反向版本：`double_top` 确认后下一根 K 线做多，3% 止盈、1.5% 止损，并用 `fp.vol_pms` 归一化持仓。

In [5]:
TAKE_PROFIT = 0.03
STOP_LOSS = 0.015


def make_two_sided_strategy(indicator_cols, open_long_raw, open_short_raw):
    """用当前指标生成完整多空策略；持仓用 fp.vol_pms 做品种/波动率尺度归一化。"""
    return (
        col
        .with_cols(indicator_cols)
        .with_cols(
            open_long_raw.fill_null(col.lit(False)).alias("open_long_raw"),
            open_short_raw.fill_null(col.lit(False)).alias("open_short_raw"),
        )
        # 指标在当前 K 线收盘后才确认，所以入场信号后移一根 K 线，避免同根 K 线偷看。
        .with_cols(
            col("open_long_raw").shift(1).expanding().fill_null(col.lit(False)).alias("open_long_sig"),
            col("open_short_raw").shift(1).expanding().fill_null(col.lit(False)).alias("open_short_sig"),
        )
        .with_cols(
            col("open_long_sig", "close").stra.exit_by_pct(TAKE_PROFIT, False).expanding().alias("take_profit_long"),
            col("open_long_sig", "close").stra.exit_by_pct(STOP_LOSS, True).expanding().alias("stop_loss_long"),
            col("open_short_sig", "close").stra.exit_by_pct(TAKE_PROFIT, True).expanding().alias("take_profit_short"),
            col("open_short_sig", "close").stra.exit_by_pct(STOP_LOSS, False).expanding().alias("stop_loss_short"),
        )
        .with_cols(
            (col("take_profit_long") | col("stop_loss_long") | col("open_short_sig"))
                .fill_null(col.lit(False))
                .alias("exit_long_sig"),
            (col("take_profit_short") | col("stop_loss_short") | col("open_long_sig"))
                .fill_null(col.lit(False))
                .alias("exit_short_sig"),
        )
        .with_cols(
            col("open_long_sig", "exit_long_sig", "open_short_sig", "exit_short_sig")
                .stra.to_hold_two_sides()
                .expanding()
                .alias("hold")
        )
        .with_cols(
            (col("hold") / col.all.fp.vol_pms()).alias("hold")
        )
        .with_cols(col("close", "hold").bt.price(fee_rate=0.0).expanding())
        .over("ticker", "ct")
        .select(
            col("pnl")
                .sum()
                .group_by(col("datetime").dt.date().alias("date"))
                .batch.sort("date")
                .with_cols(col("pnl").sum().expanding().alias("pnl_cum"))
                .select("date", "pnl", "pnl_cum")
        )
    )


def calc_strategy_stats(strategy_daily: pl.DataFrame) -> pl.DataFrame:
    return col(
        col("date").first_value().alias("start_date"),
        col("date").last_value().alias("end_date"),
        col.lit(1).sum().alias("days"),
        col("pnl").sum().alias("total_pnl"),
        col("pnl").mean().alias("mean_daily_pnl"),
        col("pnl").std().alias("std_daily_pnl"),
        (col("pnl").mean() / col("pnl").std() * col.lit(252 ** 0.5)).alias("sharpe_like"),
        col("pnl").min().alias("worst_day_pnl"),
        col("pnl").max().alias("best_day_pnl"),
    ).calc_data(strategy_daily)

indicator_cols = col("high", "low", "close").investopedia.double_top()
strategy_daily_expr = make_two_sided_strategy(
    indicator_cols,
    col("double_top"),
    col.lit(False),
)
strategy_daily = strategy_daily_expr.calc_data(raw)
strategy_stats = calc_strategy_stats(strategy_daily)

print("strategy_daily shape:", strategy_daily.shape)
strategy_stats


strategy_daily shape: (859, 3)


start_date,end_date,days,total_pnl,mean_daily_pnl,std_daily_pnl,sharpe_like,worst_day_pnl,best_day_pnl
date,date,i32,f64,f64,f64,f64,f64,f64
2022-01-04,2024-12-31,859,40.454165,0.047426,2.481548,0.303383,-8.919061,10.276105


In [6]:
strategy_daily.tail(12)


date,pnl,pnl_cum
date,f64,f64
2024-12-18,0.304081,42.427152
2024-12-19,-1.485865,40.941287
2024-12-20,-1.686258,39.255029
2024-12-21,-0.125345,39.129683
2024-12-23,-0.269031,38.860652
2024-12-24,2.193568,41.05422
2024-12-25,-1.404712,39.649508
2024-12-26,-0.15077,39.498738
2024-12-27,-2.337743,37.160995


## 7. 策略 PnL 曲线

下面用 qust monitor 同时画累计 PnL 和每日 PnL。累计曲线显示这套规则跨合约、跨日期后的整体资金变化；每日柱状图用来观察收益是否集中在少数日期。

In [7]:
pnl_dashboard = col(
    col("date", "pnl_cum")
        .monitor("strategy_pnl_cum", show_axis_label=True)
        .line(),
    col("date", "pnl")
        .monitor("strategy_daily_pnl", show_axis_label=True)
        .bar(),
).monitor.make_monitor("black").monitor.add_grid([
    ["strategy_pnl_cum"],
    ["strategy_daily_pnl"],
]).runtime()

pnl_dashboard.plot(strategy_daily, open_in_jupyter=True, auto_open=False, height=640)


## 8. 使用时的注意事项

- 技术指标只能把价格结构转成可计算规则，不等于确定性交易建议。
- 形态类指标通常需要后续 K 线确认；如果用于实时交易，应把确认延迟纳入回测。
- 参数越敏感，信号越多但噪声越大；参数越保守，信号更少但滞后更明显。
- 在多合约或多股票数据上使用时，优先写 `.over("ticker", "ct")` 或合适的分组键。